# Distribution of k for Dijet n=5036

In [ ]:
# Model selection cross check...
def model_selection(x, toy_model, seed):
    # Fit all models to the toy data and compare their AIC values
    aic_values = {}

    ROOT.RooRandom.randomGenerator().SetSeed(seed)
    toy_data = toy_model.pdf.generate(ROOT.RooArgSet(x), 5036)
    data_mean = toy_data.mean(x)

    for k in range(2, 7):
        mp = models.ModelPrimitive(
            models.ExponentialMixtureModel,
            k, data_mean=data_mean,
        )
        fit_result = fitting.fit_random_restarts(
            x, toy_data, mp,
            123,
            # 20, 5, (Old Settings)
            n_restarts, n_retries # Same settings as injection jobs
        )
        if fit_result is not None:
            n_params = 2*k-1
            aic = 2 * n_params - 2 * (-fit_result["nll"])
            aic_values[k] = aic

    return aic_values

base_seed = 42
n_toys = 100

tasks = []
for i in range(n_toys):
    seed = base_seed + i
    task = so.Task(
        model_selection,
        x, toy_model, seed
    )
    tasks.append(task)

ms_results = so.run_tasks(tasks)

In [ ]:
# Count the number of times each model was selected as the best model
model_counts = {k: 0 for k in range(2, 6)}
for result in ms_results:
    if result is not None:
        best_model = min(result, key=result.get)
        model_counts[best_model] += 1

print("Model selection counts:", model_counts)

Previous result: Model selection counts: {2: 49, 3: 49, 4: 2, 5: 0}

# Background Flexibility Proof of Concept

In [ ]:
import sys
sys.path.append("/project01/ndcms/atownse2/ExponentialMixtureModel")

import ROOT
import emm

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Get data
data_tree = emm.get_data(sort_and_index=True, tree=True)

x = ROOT.RooRealVar("x", "Diphoton Mass [GeV]", 500, 4000)
# index=ROOT.RooRealVar("index", "index", 0, 0, 1e6)

data = ROOT.RooDataSet("mgg", "mgg", ROOT.RooArgSet(x), ROOT.RooFit.Import(data_tree))

In [ ]:
# Plot fits of varying complexity to the signal + background data
models = [emm.Dijet(x)]
models += [emm.ExponentialMixtureModel(x, k, data_mean=data.mean(x)) for k in range(2, 4)]
model_labels = [m.name for m in models]
for m in models:
    m.pdf.fitTo(data, PrintLevel=-1)

emm.plot_fits(data, x, models, model_labels, logx=False)

In [ ]:
# Add signal to tail
m = 2500
s = 40
n = 4

data_sig = data.Clone()
vals_to_add = np.random.normal(m, s, size=n)
for val in vals_to_add:
    x.setVal(val)
    # index.setVal(data_sig.numEntries())
    data_sig.add(ROOT.RooArgSet(x))

print(f"Data entries (with tail events): {data_sig.numEntries()}")

# Plot fits of varying complexity to the signal + background data
models = [emm.Dijet(x)]
models += [emm.ExponentialMixtureModel(x, k, data_mean=data_sig.mean(x)) for k in [2, 3, 4]]
model_labels = [m.name for m in models]
for m in models:
    m.pdf.fitTo(data_sig, PrintLevel=-1)
emm.plot_fits(data_sig, x, models, model_labels, logx=False)

In [ ]:
# Plot fits of varying complexity to the signal + background data
models = [emm.Dijet(x)]
models += [emm.ExponentialMixtureModel(x, k, data_mean=data.mean(x)) for k in range(2, 4)]
model_labels = [m.name for m in models]
for m in models:
    m.pdf.fitTo(data, PrintLevel=-1)

emm.plot_fits(data, x, models, model_labels, logx=False)